# 03 · Graph Branch — Coordination & Swarm Detection

**AEGIS-SN** — the network half of the hybrid detector.

Notebook 02 ended on a specific weakness: a 2026 agent that writes ordinary sentences
defeats a text classifier, and the cross-generator holdout showed how quickly the text
branch degrades on an unseen model. This notebook attacks the same problem from the side
that does not care what the text says.

**The premise:** a single sophisticated agent is hard to catch. A *swarm* is not — because
coordination leaves physical traces in the interaction graph and in the timing of posts,
and those traces do not change when the attacker swaps their language model. Ten accounts
posting within the same 60-second window, on the same narrative, reciprocally amplifying
each other, is a structural signature. You cannot paraphrase your way out of it.

## What this notebook does

1. Builds a NetworkX interaction graph and characterises its topology.
2. Extracts the **coordination physics** — temporal synchrony, reciprocity, burstiness,
   circadian flatness, content reuse.
3. Trains a **features-only** classifier as the ablation floor.
4. Trains a **GraphSAGE GNN** in PyTorch Geometric.
5. Evaluates on the held-out **2026 synthetic agentic campaign**.

## ⚠ Read §7 before quoting any number from §6

Notebook 01 measured that the Cresci-2017 graph is ~98% assortative: bot–bot and
human–human edges dominate and cross-label edges are ~2% of the total. That is an artefact
of Cresci having crawled its genuine and spambot populations separately, not evidence that
the model is clever. A GNN will score near-perfectly here largely by doing community
detection. §7 quantifies how much is real and §8 is the honest evaluation.

## Dataset substitution

The spec names **TwiBot-24**. It is access-gated and not on disk, so this trains on
**Cresci-2017** with a tweet-derived co-activity graph (see notebook 01 §6). Everything
here reads `graph_model.primary_dataset` from the config — switching to TwiBot-24 when
access clears is a two-line config change, no code edits.

## 1 · Environment

In [1]:
from __future__ import annotations

import json
import math
import os
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

_here = Path.cwd()
for _candidate in (_here, *_here.parents):
    if (_candidate / "ml" / "src" / "aegis").is_dir():
        sys.path.insert(0, str(_candidate / "ml" / "src"))
        break
else:
    raise RuntimeError("Could not locate ml/src/aegis — launch Jupyter from the repo root.")

from aegis import config as acfg
from aegis import dataset_loaders as dl
from aegis import graph_features as gf
from aegis import io_utils as iou
from aegis import metrics as amx
from aegis import viz

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 220)

settings = acfg.load_config()
acfg.set_seed(settings.seed)

DEVICE = acfg.resolve_device(settings.device)
GCFG = settings.graph_model
GRAPH_NAME = GCFG.get("primary_dataset", "cresci_2017")
FBETA = float(settings.fusion_model.get("fbeta", 1.5))
WINDOW = int(GCFG.get("synchrony_window_seconds", 60))

print(f"device          : {DEVICE}")
print(f"graph dataset   : {GRAPH_NAME}")
print(f"synchrony window: {WINDOW}s")
print(f"architecture    : {GCFG.get('architecture', 'sage')}")

11:43:56 │ INFO    │ aegis │ AEGIS-SN config loaded from C:\Users\dabhi\Documents\Major-Project\Complete-project\ml\configs\default.yaml


11:43:59 │ INFO    │ aegis │ root=C:\Users\dabhi\Documents\Major-Project\Complete-project | seed=42 | smoke_test=True | device=cpu


11:43:59 │ WARNING │ aegis │ SMOKE_TEST is ON: datasets capped at 1500 rows and epochs reduced. Set AEGIS_SMOKE_TEST=0 for a publication run.


device          : cpu
graph dataset   : cresci_2017
synchrony window: 60s
architecture    : sage


## 2 · Load the graph

In [2]:
nodes = iou.load_frame(settings.paths.processed / "graph_nodes.parquet")
edges = iou.load_frame(settings.paths.processed / "graph_edges.parquet")
posts = iou.load_frame(settings.paths.processed / "graph_posts.parquet")
posts["created_at"] = pd.to_datetime(posts["created_at"], errors="coerce", utc=True)

bundle = dl.GraphBundle(
    name=GRAPH_NAME, nodes=nodes, edges=edges, posts=posts, provenance=iou.PROV_REAL,
)
print(bundle.summary())

print(f"\nedges by relation:\n{edges['relation'].value_counts().to_string()}")
print(f"\nlabel balance: {nodes['label'].value_counts().to_dict()}")
print(f"split balance:\n{nodes.groupby(['split', 'label']).size().unstack(fill_value=0).to_string()}")
print(f"\nposts: {len(posts):,} over "
      f"{posts['created_at'].min():%Y-%m-%d} → {posts['created_at'].max():%Y-%m-%d}")

{'dataset': 'cresci_2017', 'provenance': 'REAL', 'era': '', 'nodes': 2074, 'edges': 292211, 'posts': 410359, 'bot_rate': 0.4778, 'relations': ['co_hashtag', 'co_mention', 'co_reply', 'co_retweet', 'co_text', 'mentioned', 'replied_to']}

edges by relation:
relation
co_text       127600
co_mention     70716
co_hashtag     70293
co_reply       12542
co_retweet     10648
mentioned        322
replied_to        90

label balance: {0: 1083, 1: 991}
split balance:
label    0    1
split          
test   166  159
train  757  681
val    160  151

posts: 410,359 over 2009-03-19 → 2015-05-01


### The relations, and what each one means

| relation | construction | reads as |
|---|---|---|
| `replied_to` | `in_reply_to_user_id` | direct conversation |
| `mentioned` | `@handle` resolved against `screen_name` | direct address |
| `co_retweet` | both accounts retweeted the same status | **amplification ring** |
| `co_reply` | both replied to the same account | shared target |
| `co_hashtag` | both used the same rare hashtag | shared narrative frame |
| `co_mention` | both mentioned the same handle | shared target |
| `co_text` | both posted the same normalised text | **copy-paste campaign** |

The `co_*` relations dominate by volume, and that is deliberate — notebook 01 §6 showed
direct interaction alone yields 164 edges across 4,465 accounts, because Cresci's two
populations barely interact with each other. For a retweet ring the co-retweet network
*is* the campaign.

## 3 · NetworkX topology

Before any learning: what shape is this graph? Community structure is the thing to look
at, because coordinated accounts should fall into tight, dense communities while organic
accounts spread across many loose ones.

In [3]:
import networkx as nx

t0 = time.time()
G = nx.from_pandas_edgelist(
    edges, source="source", target="target", edge_attr="relation",
    create_using=nx.DiGraph(),
)
G.add_nodes_from(nodes["user_id"].astype(str))
label_map = dict(zip(nodes["user_id"].astype(str), nodes["label"].astype(int)))
nx.set_node_attributes(G, label_map, "label")

U = G.to_undirected()
print(f"built in {time.time() - t0:.1f}s")
print(f"nodes        : {G.number_of_nodes():,}")
print(f"edges        : {G.number_of_edges():,} directed / {U.number_of_edges():,} undirected")
print(f"density      : {nx.density(U):.5f}")
print(f"components   : {nx.number_connected_components(U):,}")
_giant = max(nx.connected_components(U), key=len)
print(f"giant comp.  : {len(_giant):,} nodes ({100 * len(_giant) / G.number_of_nodes():.1f}%)")
print(f"reciprocity  : {nx.overall_reciprocity(G):.4f}")
print(f"assortativity (by label): {nx.attribute_assortativity_coefficient(G, 'label'):.4f}")

built in 2.3s
nodes        : 2,074
edges        : 250,895 directed / 250,828 undirected
density      : 0.11668
components   : 11
giant comp.  : 2,064 nodes (99.5%)


reciprocity  : 0.0005
assortativity (by label): 0.9576


**Reading the assortativity coefficient.** It runs from −1 (every edge crosses classes) to
+1 (no edge ever crosses). A value near +1 is the quantitative form of the warning in the
header: the two classes barely touch, so separating them is close to trivial. Keep this
number next to §6's F1 — they should be read together, and §7 is where the ablation
disentangles them.

In [4]:
_deg = dict(U.degree())
_dd = pd.DataFrame({
    "user_id": list(_deg), "degree": list(_deg.values()),
}).assign(label=lambda d: d["user_id"].map(label_map))

print("degree by label:")
print(_dd.groupby("label")["degree"].agg(["count", "mean", "median", "std", "max"]).round(1).to_string())

try:
    import community as community_louvain  # python-louvain

    _part = community_louvain.best_partition(U, random_state=settings.seed)
    _cd = pd.DataFrame({
        "user_id": list(_part), "community": list(_part.values()),
    }).assign(label=lambda d: d["user_id"].map(label_map))
    _summary = (
        _cd.groupby("community")
        .agg(size=("user_id", "size"), bot_rate=("label", "mean"))
        .sort_values("size", ascending=False)
        .head(12).round(3)
    )
    print(f"\nLouvain: {_cd['community'].nunique()} communities, "
          f"modularity {community_louvain.modularity(_part, U):.4f}")
    print(_summary.to_string())
    print(
        "\nCommunities with bot_rate near 0.0 or 1.0 are label-pure. A long run of those"
        "\nis the assortativity above, seen from the other direction."
    )
except ImportError:
    print("\npython-louvain not installed — skipping community detection")
    _part = None

degree by label:
       count   mean  median    std  max
label                                  
0       1083  175.1   166.0  100.7  530
1        991  314.9   320.0   71.6  465



Louvain: 15 communities, modularity 0.4575
           size  bot_rate
community                
0          1103     0.026
2           952     0.999
5             5     1.000
3             2     1.000
4             2     0.000
1             1     1.000
6             1     0.000
7             1     0.000
8             1     0.000
9             1     0.000
10            1     0.000
11            1     0.000

Communities with bot_rate near 0.0 or 1.0 are label-pure. A long run of those
is the assortativity above, seen from the other direction.


## 4 · Coordination physics

`graph_features.build_features` computes the 16 features in `graph_model.features`. They
fall into four families, and the reasoning behind each matters more than the arithmetic:

**Temporal — the hardest to fake**
* `synchrony_score` — how often this account posts within `window_seconds` of another,
  measured *against a null model* of its own posting rate. Raw co-occurrence counts would
  just rank the most prolific accounts; the lift over chance is what identifies
  coordination. This is usually the single strongest feature.
* `burstiness` — Goh & Barabási's B. Humans are bursty (silence, then a flurry); schedulers
  are regular. B→+1 bursty, B→0 Poisson, B→−1 metronomic.
* `circadian_flatness` — humans sleep. An account with a flat 24-hour histogram either
  never sleeps or is several people, and both are worth a look.
* `posting_entropy` — Shannon entropy over the hour-of-day histogram.
* `memory_coefficient` — autocorrelation of consecutive inter-post intervals.

**Structural**
* `reciprocity` — local edge reciprocity. Mutual-amplification rings run high.
* `clustering_coefficient`, `in_degree`, `out_degree`, `degree_ratio`.

**Content**
* `content_duplication_ratio` — near-duplicate posts *within* the account.
* `cross_account_dup_ratio` — near-duplicates *shared with neighbours*. This is the direct
  copy-paste signal.
* `hashtag_jaccard_mean` — co-hashtag overlap with neighbours.

**Profile**
* `account_age_days`, `followers_to_following`.

In [5]:
t0 = time.time()
features = gf.build_features(bundle, settings, window_seconds=WINDOW)
print(f"computed in {time.time() - t0:.1f}s")
print(f"matrix: {features.features.shape}  ({len(features.feature_columns)} features)")
print(f"note  : {features.note}")

features.features.loc[:, list(features.feature_columns)].describe().T.round(3)

11:44:11 │ INFO    │ aegis.graph │ features for cresci_2017: 2074 nodes, 292211 edges, 410359 posts (window=60s)


11:44:42 │ WARNING │ aegis.graph │ synchrony: 60 time bucket(s) exceeded 400 accounts and were subsampled. Those are platform-wide events, not swarms — but the subsampling does make the score for accounts inside them noisier.


11:45:06 │ WARNING │ aegis.graph │ content features: 228511 distinct posts — skipping the near-duplicate pass and using exact duplicates only. Coordinated accounts that paraphrase will be under-scored.


computed in 57.0s
matrix: (2074, 21)  (16 features)
note  : cresci_2017 (REAL) window=60s min_events=3


,count,mean,std,min,25%,50%,75%,max
synchrony_score,2074.0,0.931,0.181,0.000,0.946,0.974,0.993,1.000
synchrony_partner_count,2074.0,641.849,264.749,0.000,470.250,729.000,834.000,1514.000
reciprocity,2074.0,0.001,0.013,0.000,0.000,0.000,0.000,0.333
clustering_coefficient,2074.0,0.387,0.105,0.000,0.345,0.359,0.397,1.000
in_degree,2074.0,140.892,108.873,0.000,48.000,120.000,215.750,610.000
out_degree,2074.0,140.892,109.456,0.000,48.000,117.500,219.000,643.000
degree_ratio,2074.0,4.452,17.385,0.000,0.311,0.972,2.882,402.000
burstiness,2074.0,0.444,0.136,-0.254,0.357,0.479,0.530,0.851
memory_coefficient,2074.0,0.228,0.198,-0.531,0.047,0.217,0.393,0.723
posting_entropy,2074.0,0.840,0.073,0.173,0.836,0.861,0.873,0.982


### Does any of it separate the classes?

`separation_report` gives Cohen's *d* per feature. This is the go/no-go check, and it is
worth running before a 200-epoch training run rather than after.

Rough convention: |d| ≥ 0.2 small, ≥ 0.5 medium, ≥ 0.8 large. If `synchrony_score` and
`cross_account_dup_ratio` are flat, the graph branch has nothing and no amount of GNN
capacity will conjure it.

In [6]:
separation = features.separation_report()
print(separation.to_string(index=False))

_strong = separation[separation["cohens_d"].abs() >= 0.8]
print(f"\n{len(_strong)} features with a large effect (|d| >= 0.8):")
print(f"  {', '.join(_strong['feature'].tolist()) or 'none'}")

viz.plot_feature_importance(
    separation["feature"].tolist(), separation["cohens_d"].abs().tolist(),
    top_n=16, save_as=settings.paths.figures / "03_feature_separation.png",
)

                  feature  bot_mean  human_mean  cohens_d
  cross_account_dup_ratio    0.9517      0.0777     5.783
       memory_coefficient    0.3825      0.0875     2.214
               burstiness    0.5024      0.3899     0.918
                in_degree  175.2109    109.4894     0.631
               out_degree  173.3693    111.1745     0.591
         account_age_days 5016.6418   5317.1283    -0.567
   clustering_coefficient    0.3578      0.4130    -0.554
          posting_entropy    0.8588      0.8236     0.504
       circadian_flatness    0.5216      0.5772    -0.442
  synchrony_partner_count  609.0293    671.8800    -0.241
content_duplication_ratio    0.0020      0.0105    -0.217
   followers_to_following   -0.0091      0.1299    -0.159
              reciprocity    0.0005      0.0024    -0.149
          synchrony_score    0.9369      0.9248     0.068
             degree_ratio    4.9038      4.0391     0.049
     hashtag_jaccard_mean    0.0000      0.0000     0.000

3 features wi

11:45:08 │ INFO    │ aegis.viz │ figure -> C:\Users\dabhi\Documents\Major-Project\Complete-project\reports\figures\03_feature_separation.png


<Axes: title={'center': 'Top 16 features'}, xlabel='importance'>

### Synchrony in detail — and a parameter that is wrong for this corpus

`synchrony_score` is the feature the design leans on hardest, so it gets its own look.

**On Cresci it saturates, and the run below shows it.** Expect both classes to sit near
1.0 with a Cohen's *d* close to zero, and the bot rate among the top-100 most synchronous
accounts to come out *below* the base rate — synchrony is mildly **anti**-correlated with
the label here.

That is a measurement artefact with a specific cause. Cresci spans **six years**
(2009–2015) at ~200 posts per account. With a 60-second window over that span, virtually
every pair of active accounts co-occurs in *some* bucket, so the statistic tops out and
stops discriminating. The warning the feature builder emits — a number of time buckets
exceeding 400 accounts and being subsampled — is the same problem seen from the other
side: those buckets are platform-wide events, not swarms.

`synchrony_window_seconds: 60` is calibrated for a **campaign** observed over hours or
days, which is what the synthetic 2026 corpus and any live deployment look like. It is the
wrong parameter for a six-year archival crawl. Two honest options, and the notebook takes
the first:

1. Report it, and let `cross_account_dup_ratio` and `memory_coefficient` carry Cresci —
   which, per the separation table, they comfortably do.
2. Re-scope synchrony to a rolling window per active period. Correct, and out of scope for
   this notebook.

The feature is retained rather than dropped because it is one of the strongest signals on
the campaign corpus, which is the deployment-realistic one.

In [7]:
_f = features.features
print("synchrony_score by label:")
print(_f.groupby("label")["synchrony_score"].describe().round(3).to_string())

print("\ntop 10 most synchronous accounts:")
print(
    _f.nlargest(10, "synchrony_score")
    .loc[:, ["user_id", "label", "synchrony_score", "synchrony_partner_count",
             "burstiness", "circadian_flatness", "cross_account_dup_ratio"]]
    .round(3).to_string(index=False)
)

_topk = _f.nlargest(100, "synchrony_score")
print(f"\nbot rate in the top-100 by synchrony: {_topk['label'].mean():.3f} "
      f"(base rate {_f['label'].mean():.3f})")

synchrony_score by label:
        count   mean    std  min    25%    50%    75%  max
label                                                     
0      1083.0  0.925  0.235  0.0  0.977  0.992  0.997  1.0
1       991.0  0.937  0.090  0.0  0.927  0.956  0.972  1.0

top 10 most synchronous accounts:
   user_id  label  synchrony_score  synchrony_partner_count  burstiness  circadian_flatness  cross_account_dup_ratio
 497404180      0            1.000                     72.0       0.510               0.067                    0.555
  17436585      0            1.000                     72.0       0.488               0.754                    0.030
  14980820      0            1.000                     72.0       0.477               0.862                    0.005
 859013762      1            1.000                    787.0       0.191               0.926                    0.005
1536537319      1            1.000                    972.0       0.233               0.666                    0.000
1

## 5 · Ablation floor — features only, no graph structure

**This is the control the whole notebook turns on.** If a plain gradient-boosted tree over
the 16 features scores as well as the GNN, then message passing contributed nothing and
the "GNN" is decoration on a tabular model. Given the assortativity, that is a live
possibility and it has to be measured, not assumed.

Note the features are not structure-free — `in_degree`, `reciprocity` and
`clustering_coefficient` are computed *from* the graph. What this ablation isolates is the
value of **message passing**: does aggregating a neighbourhood's features beat looking at
each node alone?

In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

X_scaled, scaler = gf.scale_features(features, train_split="train")
y = features.y
train_mask = features.mask("train")
val_mask = features.mask("val")
test_mask = features.mask("test")

print(f"train {train_mask.sum():,} | val {val_mask.sum():,} | test {test_mask.sum():,}")

tabular: dict[str, amx.ClassificationReport] = {}

_lr = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=settings.seed)
_lr.fit(X_scaled[train_mask], y[train_mask])
tabular["logreg_features"] = amx.evaluate(
    y[test_mask], _lr.predict_proba(X_scaled[test_mask])[:, 1], beta=FBETA
)

_rf = RandomForestClassifier(
    n_estimators=400, class_weight="balanced", n_jobs=-1, random_state=settings.seed,
)
_rf.fit(X_scaled[train_mask], y[train_mask])
rf_scores_test = _rf.predict_proba(X_scaled[test_mask])[:, 1]
tabular["random_forest_features"] = amx.evaluate(y[test_mask], rf_scores_test, beta=FBETA)

try:
    from lightgbm import LGBMClassifier

    _lgb = LGBMClassifier(
        n_estimators=400, learning_rate=0.05, class_weight="balanced",
        random_state=settings.seed, verbose=-1,
    )
    _lgb.fit(X_scaled[train_mask], y[train_mask])
    tabular["lightgbm_features"] = amx.evaluate(
        y[test_mask], _lgb.predict_proba(X_scaled[test_mask])[:, 1], beta=FBETA
    )
except ImportError:
    print("lightgbm not installed — skipping")

print(amx.compare_reports(tabular).to_string())

print("\nRandomForest feature importance:")
_imp = pd.Series(_rf.feature_importances_, index=list(features.feature_columns)).sort_values(ascending=False)
print(_imp.round(4).to_string())
viz.plot_feature_importance(
    _imp.index.tolist(), _imp.values.tolist(), top_n=16,
    save_as=settings.paths.figures / "03_rf_importance.png",
)

train 1,438 | val 311 | test 325


                        precision    recall        f1  delta_f1     fbeta   roc_auc    pr_auc       mcc     brier  accuracy  alert_rate  threshold  support  n_positive   tn  fp  fn   tp
model                                                                                                                                                                                    
lightgbm_features        0.987013  0.955975  0.971246  0.000000  0.965315  0.995946  0.996435  0.945001  0.024581  0.972308    0.473846        0.5      325         159  164   2   7  152
random_forest_features   1.000000  0.930818  0.964169 -0.007077  0.951063  0.994753  0.995267  0.934329  0.024422  0.966154    0.455385        0.5      325         159  166   0  11  148
logreg_features          0.986577  0.924528  0.954545 -0.016701  0.942773  0.978632  0.984970  0.915433  0.037081  0.956923    0.458462        0.5      325         159  164   2  12  147

RandomForest feature importance:
cross_account_dup_ratio      0.4203


11:45:12 │ INFO    │ aegis.viz │ figure -> C:\Users\dabhi\Documents\Major-Project\Complete-project\reports\figures\03_rf_importance.png


<Axes: title={'center': 'Top 16 features'}, xlabel='importance'>

## 6 · GraphSAGE

`to_pyg_data` converts the bundle into a PyG `Data` object with `train/val/test` node
masks. The graph is treated as undirected for message passing — a co-retweet relation is
inherently symmetric, and for the directed relations the *fact* of interaction matters more
to coordination than its direction.

**GraphSAGE rather than GCN**, because SAGE's sampled-neighbour aggregation is inductive:
it can score an account it has never seen, which is what the FastAPI service in Phase 2
needs to do on live traffic. A transductive GCN would have to be retrained for every new
account.

In [9]:
try:
    import torch
    import torch.nn.functional as F
    from torch_geometric.nn import GATConv, SAGEConv

    HAS_PYG = True
except ImportError as exc:
    HAS_PYG = False
    print(f"PyTorch Geometric unavailable ({exc}).")
    print("Install:  pip install torch-geometric")
    print("The tabular ablation in §5 still ran; the GNN sections will be skipped.")

In [10]:
if HAS_PYG:
    data = gf.to_pyg_data(features, scaled=X_scaled, undirected=True)
    print(data)
    print(f"features per node: {data.num_node_features}")
    print(f"train/val/test   : {int(data.train_mask.sum())} / "
          f"{int(data.val_mask.sum())} / {int(data.test_mask.sum())}")


# The conditional base keeps this cell runnable when PyG is absent: the expression
# short-circuits before touching `torch`, so the class still defines and the notebook
# reaches the tabular results instead of dying on an import.
class SwarmGNN(torch.nn.Module if HAS_PYG else object):
    """
    Two-layer GraphSAGE (or GAT) node classifier.

    Two layers, not more: each layer widens the receptive field by one hop, and with a
    graph this dense a third hop reaches most of the component and the embeddings collapse
    toward each other. That is over-smoothing, and it shows up as *worse* validation F1
    with *more* capacity — the failure that looks like a bug and is not.
    """

    def __init__(self, in_channels: int, hidden: int, *, architecture: str = "sage",
                 heads: int = 4, dropout: float = 0.4, num_layers: int = 2):
        super().__init__()
        self.dropout = dropout
        self.convs = torch.nn.ModuleList()
        self.norms = torch.nn.ModuleList()

        for layer in range(num_layers):
            src = in_channels if layer == 0 else hidden
            if architecture == "gat":
                # concat=False averages the heads so the output width stays `hidden`,
                # which keeps the layer stack uniform.
                self.convs.append(GATConv(src, hidden, heads=heads, concat=False, dropout=dropout))
            else:
                self.convs.append(SAGEConv(src, hidden))
            self.norms.append(torch.nn.BatchNorm1d(hidden))

        self.head = torch.nn.Linear(hidden, 2)

    def forward(self, x, edge_index):
        for conv, norm in zip(self.convs, self.norms):
            x = conv(x, edge_index)
            x = norm(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return self.head(x)

Data(x=[2074, 16], edge_index=[2, 584422], y=[2074], train_mask=[2074], val_mask=[2074], test_mask=[2074], user_id=[2074])
features per node: 16
train/val/test   : 1438 / 311 / 325


In [11]:
if HAS_PYG:
    torch.manual_seed(settings.seed)
    EPOCHS = 50 if settings.smoke_test else int(GCFG.get("epochs", 200))
    PATIENCE = int(GCFG.get("early_stopping_patience", 30))

    device = torch.device(DEVICE if DEVICE != "mps" else "cpu")
    data = data.to(device)

    model = SwarmGNN(
        in_channels=data.num_node_features,
        hidden=int(GCFG.get("hidden_channels", 128)),
        architecture=str(GCFG.get("architecture", "sage")),
        heads=int(GCFG.get("heads", 4)),
        dropout=float(GCFG.get("dropout", 0.4)),
        num_layers=int(GCFG.get("num_layers", 2)),
    ).to(device)

    optimiser = torch.optim.Adam(
        model.parameters(),
        lr=float(GCFG.get("learning_rate", 5e-3)),
        weight_decay=float(GCFG.get("weight_decay", 5e-4)),
    )

    # Same class-weighting rationale as notebook 02: reweight the loss rather than
    # resample the nodes, because resampling a graph means deleting edges.
    _counts = np.bincount(y[train_mask], minlength=2)
    _w = torch.tensor(
        (_counts.sum() / (2.0 * np.maximum(_counts, 1))), dtype=torch.float, device=device
    )
    print(f"class weights: {_w.tolist()}")
    print(f"epochs={EPOCHS} hidden={GCFG.get('hidden_channels')} "
          f"arch={GCFG.get('architecture')} params="
          f"{sum(p.numel() for p in model.parameters()):,}")

class weights: [0.9498018622398376, 1.0558003187179565]
epochs=50 hidden=128 arch=sage params=37,890


In [12]:
if HAS_PYG:
    history: list[dict] = []
    best_f1, best_state, since_improved = -1.0, None, 0
    t0 = time.time()

    for epoch in range(1, EPOCHS + 1):
        model.train()
        optimiser.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask], weight=_w)
        loss.backward()
        optimiser.step()

        model.eval()
        with torch.no_grad():
            logits = model(data.x, data.edge_index)
            probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()

        val_rep = amx.evaluate(
            data.y[data.val_mask].cpu().numpy(), probs[data.val_mask.cpu().numpy()], beta=FBETA
        )
        history.append({
            "epoch": epoch, "loss": float(loss), "eval_f1": val_rep.f1,
            "eval_precision": val_rep.precision, "eval_recall": val_rep.recall,
            "eval_roc_auc": val_rep.roc_auc,
        })

        if val_rep.f1 > best_f1:
            best_f1, since_improved = val_rep.f1, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            since_improved += 1
            if since_improved >= PATIENCE:
                print(f"early stop at epoch {epoch} (no val F1 gain in {PATIENCE})")
                break

        if epoch % 10 == 0 or epoch == 1:
            print(f"  epoch {epoch:>3}  loss {float(loss):.4f}  "
                  f"val F1 {val_rep.f1:.4f}  val AUC {val_rep.roc_auc:.4f}")

    if best_state is not None:
        model.load_state_dict(best_state)
    print(f"\ntrained in {time.time() - t0:.1f}s — best val F1 {best_f1:.4f}")

    viz.plot_training_curve(history, save_as=settings.paths.figures / "03_gnn_training.png")

  epoch   1  loss 0.7033  val F1 0.9728  val AUC 0.9742


  epoch  10  loss 0.0574  val F1 0.9763  val AUC 0.9882


  epoch  20  loss 0.0343  val F1 0.9900  val AUC 0.9912


  epoch  30  loss 0.0268  val F1 0.9900  val AUC 0.9925


  epoch  40  loss 0.0197  val F1 0.9900  val AUC 0.9930


early stop at epoch 47 (no val F1 gain in 30)

trained in 43.5s — best val F1 0.9900


11:46:01 │ INFO    │ aegis.viz │ figure -> C:\Users\dabhi\Documents\Major-Project\Complete-project\reports\figures\03_gnn_training.png


In [13]:
if HAS_PYG:
    model.eval()
    with torch.no_grad():
        gnn_probs = torch.softmax(model(data.x, data.edge_index), dim=-1)[:, 1].cpu().numpy()

    _val_mask_np = data.val_mask.cpu().numpy()
    _test_mask_np = data.test_mask.cpu().numpy()

    graph_threshold, _gval = amx.tune_threshold(
        y[_val_mask_np], gnn_probs[_val_mask_np], objective="fbeta", beta=FBETA
    )
    gnn_report = amx.evaluate(
        y[_test_mask_np], gnn_probs[_test_mask_np], threshold=graph_threshold, beta=FBETA
    )

    print(f"tuned threshold : {graph_threshold:.3f}")
    print(f"GraphSAGE test  : {gnn_report}")
    print(f"\n{amx.confusion_frame(gnn_report).to_string()}")

    for _m in ("f1", "recall", "roc_auc"):
        _pt, _lo, _hi = amx.bootstrap_ci(
            y[_test_mask_np], gnn_probs[_test_mask_np], metric=_m,
            threshold=graph_threshold, seed=settings.seed,
        )
        print(f"  {_m:<8} {_pt:.4f}  95% CI [{_lo:.4f}, {_hi:.4f}]")

    viz.plot_roc_pr(y[_test_mask_np], gnn_probs[_test_mask_np],
                    save_as=settings.paths.figures / "03_gnn_roc_pr.png")
    viz.plot_score_distributions(y[_test_mask_np], gnn_probs[_test_mask_np],
                                 threshold=graph_threshold,
                                 save_as=settings.paths.figures / "03_gnn_scores.png")
    viz.plot_threshold_sweep(
        y[_val_mask_np], gnn_probs[_val_mask_np], beta=FBETA, chosen=graph_threshold,
        save_as=settings.paths.figures / "03_gnn_threshold_sweep.png",
    )

11:46:02 │ INFO    │ aegis.metrics │ tune_threshold(fbeta, beta=1.50): 0.5000 -> fbeta=0.9862 (0.5 would give 0.9862)


tuned threshold : 0.500
GraphSAGE test  : <report P=0.974 R=0.956 F1=0.965 F1.5=0.962 AUC=0.996 AP=0.996 @thr=0.50 n=325>

predicted            pred human/benign  pred adversarial  total
actual                                                         
actual human/benign                162                 4    166
actual adversarial                   7               152    159
total                              169               156    325


  f1       0.9651  95% CI [0.9438, 0.9844]


  recall   0.9560  95% CI [0.9231, 0.9869]


  roc_auc  0.9956  95% CI [0.9906, 0.9991]


11:46:05 │ INFO    │ aegis.viz │ figure -> C:\Users\dabhi\Documents\Major-Project\Complete-project\reports\figures\03_gnn_roc_pr.png


11:46:05 │ INFO    │ aegis.viz │ figure -> C:\Users\dabhi\Documents\Major-Project\Complete-project\reports\figures\03_gnn_scores.png


11:46:06 │ INFO    │ aegis.viz │ figure -> C:\Users\dabhi\Documents\Major-Project\Complete-project\reports\figures\03_gnn_threshold_sweep.png


## 7 · Ablation — what did the graph actually buy?

The comparison the header promised. Three rows:

* **RandomForest on features** — no message passing.
* **GraphSAGE** — message passing over the same features.
* **GraphSAGE on a degree-preserving rewired graph** — the features are unchanged and the
  degree sequence is unchanged, but the edges are randomised, so any *real* community
  structure is destroyed while the graph statistics are preserved.

The third row is the one that settles it. If the rewired GNN scores nearly as well as the
real one, then the model was reading node features and the topology was decorative. If it
collapses, the structure carried genuine information.

In [14]:
if HAS_PYG:
    ablation = dict(tabular)
    ablation["graphsage"] = gnn_report

    _ei = data.edge_index.cpu().numpy()
    _rng = np.random.default_rng(settings.seed)
    # Shuffling the destination endpoints preserves each node's out-degree exactly and its
    # in-degree in distribution, so the graph stays the same "size and shape" while the
    # community structure is destroyed.
    _shuffled = _ei.copy()
    _shuffled[1] = _rng.permutation(_shuffled[1])
    _rewired = torch.tensor(_shuffled, dtype=torch.long, device=device)

    torch.manual_seed(settings.seed)
    _null = SwarmGNN(
        in_channels=data.num_node_features,
        hidden=int(GCFG.get("hidden_channels", 128)),
        architecture=str(GCFG.get("architecture", "sage")),
        dropout=float(GCFG.get("dropout", 0.4)),
        num_layers=int(GCFG.get("num_layers", 2)),
    ).to(device)
    _opt = torch.optim.Adam(_null.parameters(), lr=float(GCFG.get("learning_rate", 5e-3)),
                            weight_decay=float(GCFG.get("weight_decay", 5e-4)))
    for _ in range(min(EPOCHS, 60)):
        _null.train()
        _opt.zero_grad()
        _o = _null(data.x, _rewired)
        F.cross_entropy(_o[data.train_mask], data.y[data.train_mask], weight=_w).backward()
        _opt.step()

    _null.eval()
    with torch.no_grad():
        _np_probs = torch.softmax(_null(data.x, _rewired), dim=-1)[:, 1].cpu().numpy()
    ablation["graphsage_rewired_null"] = amx.evaluate(
        y[_test_mask_np], _np_probs[_test_mask_np], threshold=graph_threshold, beta=FBETA
    )

    print(amx.compare_reports(ablation).to_string())
    _lift = gnn_report.f1 - ablation["graphsage_rewired_null"].f1
    _lift_vs_rf = gnn_report.f1 - tabular["random_forest_features"].f1
    print(f"\nF1 lift from real topology vs rewired null : {_lift:+.4f}")
    print(f"F1 lift from message passing vs RandomForest: {_lift_vs_rf:+.4f}")
    print(
        "\nA large first number means the graph structure is doing real work. A small"
        "\nsecond number means the *message passing* added little beyond the features —"
        "\nwhich, on a graph this assortative, is a perfectly plausible outcome and one"
        "\nworth reporting rather than hiding."
    )

                        precision    recall        f1  delta_f1     fbeta   roc_auc    pr_auc       mcc     brier  accuracy  alert_rate  threshold  support  n_positive   tn  fp  fn   tp
model                                                                                                                                                                                    
lightgbm_features        0.987013  0.955975  0.971246  0.000000  0.965315  0.995946  0.996435  0.945001  0.024581  0.972308    0.473846        0.5      325         159  164   2   7  152
graphsage                0.974359  0.955975  0.965079 -0.006167  0.961557  0.995567  0.995957  0.932409  0.022776  0.966154    0.480000        0.5      325         159  162   4   7  152
graphsage_rewired_null   0.986842  0.943396  0.964630 -0.006616  0.956351  0.988065  0.991261  0.933082  0.032543  0.966154    0.467692        0.5      325         159  164   2   9  150
random_forest_features   1.000000  0.930818  0.964169 -0.007077  0.951

## 8 · The honest test — the 2026 synthetic agentic campaign

Everything so far is Cresci-2017: 2014 Italian spambots, two disjoint crawls, ~98%
assortative. The model has never seen the thing this project is actually about.

The synthetic campaign from notebook 01 is the out-of-distribution evaluation, and it is
deliberately much harder:

* 8 LLM agents against **40 organic decoys posting on the same topic**, so there is no
  topical shortcut.
* Agents and humans are **interleaved in one interaction graph** — the two populations
  genuinely touch, unlike Cresci.
* The coordination signal is temporal and structural, not lexical.

**This is the number to quote for the graph branch.** It is measured on generated data, so
it is a simulation result and must be labelled as such — but it is a simulation of the
right threat, whereas Cresci is a real measurement of the wrong one.

In [15]:
campaign_nodes = iou.load_frame(settings.paths.processed / "campaign_nodes.parquet")
campaign_edges = iou.load_frame(settings.paths.processed / "campaign_edges.parquet")
campaign_posts = iou.load_frame(settings.paths.processed / "campaign_posts.parquet")
campaign_posts["created_at"] = pd.to_datetime(campaign_posts["created_at"], errors="coerce", utc=True)

campaign_bundle = dl.GraphBundle(
    name="synthetic_campaign", nodes=campaign_nodes, edges=campaign_edges,
    posts=campaign_posts, provenance=iou.PROV_GENERATED,
)
print(campaign_bundle.summary())

campaign_features = gf.build_features(campaign_bundle, settings, window_seconds=WINDOW)
print(f"\nfeature matrix: {campaign_features.features.shape}")
print("\nseparation on the campaign:")
print(campaign_features.separation_report().head(10).to_string(index=False))

{'dataset': 'synthetic_campaign', 'provenance': 'GENERATED', 'era': '', 'nodes': 48, 'edges': 289, 'posts': 327, 'bot_rate': 0.1667, 'relations': ['following', 'mentioned', 'replied_to', 'retweeted']}
11:46:35 │ INFO    │ aegis.graph │ features for synthetic_campaign: 48 nodes, 289 edges, 327 posts (window=60s)



feature matrix: (48, 21)

separation on the campaign:
                  feature  bot_mean  human_mean  cohens_d
          synchrony_score    0.8063      0.0000    10.243
               out_degree   18.5000      3.5250     4.618
   followers_to_following   -2.8363     -0.1702    -4.567
  synchrony_partner_count    4.0000      0.0000     3.191
         account_age_days  180.0000   2622.2250    -2.887
             degree_ratio    1.5312      0.6688     2.399
                in_degree   11.3750      4.9500     2.389
               burstiness    0.1779     -0.1095     1.971
   clustering_coefficient    0.3856      0.2348     1.098
content_duplication_ratio    0.0651      0.0000     0.710


### Score it with the models trained on Cresci

No retraining, no fine-tuning. This is a genuine transfer test: does a detector trained on
2014 Italian spambots recognise a 2026 LLM agent swarm?

In [16]:
_Xc = campaign_features.features.loc[:, list(features.feature_columns)].to_numpy(dtype=np.float32)
_Xc = scaler.transform(_Xc)
_yc = campaign_features.y

campaign_results: dict[str, amx.ClassificationReport] = {}
campaign_results["random_forest_features"] = amx.evaluate(
    _yc, _rf.predict_proba(_Xc)[:, 1], beta=FBETA
)

if HAS_PYG:
    _cdata = gf.to_pyg_data(campaign_features, scaled=_Xc, undirected=True).to(device)
    model.eval()
    with torch.no_grad():
        _cprobs = torch.softmax(model(_cdata.x, _cdata.edge_index), dim=-1)[:, 1].cpu().numpy()
    campaign_results["graphsage"] = amx.evaluate(
        _yc, _cprobs, threshold=graph_threshold, beta=FBETA
    )

print("transfer to the 2026 synthetic campaign (no retraining):")
print(amx.compare_reports(campaign_results).to_string())

if HAS_PYG:
    print(f"\nCresci test  F1 : {gnn_report.f1:.4f}")
    print(f"Campaign     F1 : {campaign_results['graphsage'].f1:.4f}")
    print(f"transfer drop   : {gnn_report.f1 - campaign_results['graphsage'].f1:+.4f}")

transfer to the 2026 synthetic campaign (no retraining):
                        precision  recall        f1  delta_f1    fbeta  roc_auc    pr_auc       mcc     brier  accuracy  alert_rate  threshold  support  n_positive  tn  fp  fn  tp
model                                                                                                                                                                              
random_forest_features   0.111111   0.125  0.117647  0.000000  0.12037  0.41875  0.155110 -0.071611  0.213503  0.687500      0.1875        0.5       48           8  32   8   7   1
graphsage                0.000000   0.000  0.000000 -0.117647  0.00000  0.57500  0.193667  0.000000  0.166667  0.833333      0.0000        0.5       48           8  40   0   8   0

Cresci test  F1 : 0.9651
Campaign     F1 : 0.0000
transfer drop   : +0.9651


### ⚠ Zero-shot transfer fails, and the reason is diagnosable

The measured result is that a detector trained on Cresci-2017 transfers to the 2026
campaign at **roughly chance or worse**. That is a negative result and it is reported as
one — but it is a negative result with an identifiable cause, and the cause is not "the
coordination features do not work".

**The features are not scale-invariant.** Look at what discriminates in each corpus:

| Cresci-2017 (2,074 nodes, 292k edges) | 2026 campaign (48 nodes, 285 edges) |
|---|---|
| `cross_account_dup_ratio` d = 5.8 | `out_degree` d = 5.6 |
| `memory_coefficient` d = 2.2 | `followers_to_following` d = −4.6 |
| `burstiness` d = 0.9 | `account_age_days` d = −2.9 |

`in_degree` and `out_degree` are **raw counts**. A well-connected account in a 48-node
graph has a degree of ~18; a poorly-connected one in Cresci has ~110. The scaler was fitted
on Cresci, so every campaign account lands in the far-left tail of that distribution and
the model reads the whole population as "isolated, therefore human". The sign of the
`roc_auc` — *below* 0.5 — is the signature of exactly that kind of systematic inversion
rather than of noise.

The fix is degree **normalisation** (degree centrality, or degree/√|E|) so the features
describe a node's position relative to its own graph. That is a change to
`graph_features.compute_structural_features` and is left as the clearly-signposted next
step rather than quietly patched here.

The section below separates the two hypotheses that the transfer number confounds:
*"the features carry no signal on agentic campaigns"* versus *"the features carry signal
but do not transfer across graph scale"*. Training and testing **within** the campaign
answers it.

In [17]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict

_yc_arr = np.asarray(_yc)
_n_pos = int(_yc_arr.sum())
_folds = int(min(5, _n_pos, len(_yc_arr) - _n_pos))

if _folds >= 2:
    # Unscaled features, refitted within the campaign: this asks only whether the
    # 16 coordination features separate agents from organic accounts *in this graph*,
    # with no cross-corpus scale mismatch in the way.
    _Xc_raw = campaign_features.features.loc[
        :, list(features.feature_columns)
    ].to_numpy(dtype=np.float32)

    _cv = StratifiedKFold(n_splits=_folds, shuffle=True, random_state=settings.seed)
    _in_domain = cross_val_predict(
        RandomForestClassifier(
            n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=settings.seed,
        ),
        _Xc_raw, _yc_arr, cv=_cv, method="predict_proba",
    )[:, 1]

    _thr_c, _ = amx.tune_threshold(_yc_arr, _in_domain, objective="fbeta", beta=FBETA)
    campaign_results["random_forest_in_domain_cv"] = amx.evaluate(
        _yc_arr, _in_domain, threshold=_thr_c, beta=FBETA
    )

    print(f"in-domain {_folds}-fold CV on the campaign ({len(_yc_arr)} accounts, "
          f"{_n_pos} agents):")
    print(f"  {campaign_results['random_forest_in_domain_cv']}")
    print(
        "\nIf this is high while the zero-shot transfer above is at chance, the"
        "\nconclusion is that the coordination features DO separate a 2026 agent swarm —"
        "\nthey simply do not survive being scaled against a graph 40x larger. That is a"
        "\nfixable engineering defect, not a failure of the premise."
    )
    if campaign_results["random_forest_in_domain_cv"].f1 > 0.98:
        print(
            f"\n*** CAVEAT: a near-perfect score on {_n_pos} agents is NOT a strong result."
            "\n*** With this few positives a single fold holds one or two of them, so the"
            "\n*** metric has almost no resolution and a confidence interval would span most"
            "\n*** of [0, 1]. It also says the generator may be producing agents that are too"
            "\n*** easy: every agent shares a persona template, a posting schedule and a"
            "\n*** hashtag set, so they are more homogeneous than a real swarm would be."
            "\n*** Read this as 'the features are not broken', NOT as a detection rate."
            "\n*** To make it a real number: raise synthetic.n_agents and n_human_decoys,"
            "\n*** widen posts_per_agent_per_turn, and add organic accounts that legitimately"
            "\n*** co-post on the campaign hashtags."
        )
else:
    print(f"only {_n_pos} agents — too few to cross-validate within the campaign")

print("\nall campaign evaluations:")
print(amx.compare_reports(campaign_results).to_string())
print(
    "\nAgainst notebook 02 §8: the project's claim is that coordination structure survives"
    "\nan era and generator change better than text style does. On the evidence here that"
    "\nclaim holds only for scale-invariant features, and the raw degree features must be"
    "\nnormalised before the claim can be made cleanly."
)

11:46:40 │ INFO    │ aegis.metrics │ tune_threshold(fbeta, beta=1.50): 0.1644 -> fbeta=1.0000 (0.5 would give 1.0000)


in-domain 5-fold CV on the campaign (48 accounts, 8 agents):
  <report P=1.000 R=1.000 F1=1.000 F1.5=1.000 AUC=1.000 AP=1.000 @thr=0.16 n=48>

If this is high while the zero-shot transfer above is at chance, the
conclusion is that the coordination features DO separate a 2026 agent swarm —
they simply do not survive being scaled against a graph 40x larger. That is a
fixable engineering defect, not a failure of the premise.

*** CAVEAT: a near-perfect score on 8 agents is NOT a strong result.
*** With this few positives a single fold holds one or two of them, so the
*** metric has almost no resolution and a confidence interval would span most
*** of [0, 1]. It also says the generator may be producing agents that are too
*** easy: every agent shares a persona template, a posting schedule and a
*** hashtag set, so they are more homogeneous than a real swarm would be.
*** Read this as 'the features are not broken', NOT as a detection rate.
*** To make it a real number: raise synthetic.n_age

In [18]:
_cf = campaign_features.features.copy()
if HAS_PYG:
    _cf["score"] = _cprobs
    print("agent accounts, ranked by swarm probability:")
    print(
        _cf[_cf["label"] == 1]
        .nlargest(10, "score")
        .loc[:, ["user_id", "score", "synchrony_score", "synchrony_partner_count",
                 "reciprocity", "circadian_flatness", "cross_account_dup_ratio"]]
        .round(3).to_string(index=False)
    )
    print("\nhighest-scoring ORGANIC accounts (these are the false positives):")
    print(
        _cf[_cf["label"] == 0]
        .nlargest(5, "score")
        .loc[:, ["user_id", "score", "synchrony_score", "burstiness", "circadian_flatness"]]
        .round(3).to_string(index=False)
    )

agent accounts, ranked by swarm probability:
                 user_id  score  synchrony_score  synchrony_partner_count  reciprocity  circadian_flatness  cross_account_dup_ratio
  agent::orchestrator_00    0.0            0.691                      5.0        0.400               0.633                    0.217
agent::honest_reviews_uk    0.0            0.777                      1.0        0.538               0.509                    0.000
 agent::jord_wasdoubtful    0.0            0.944                      6.0        0.545               0.733                    0.111
   agent::liftwithcallum    0.0            0.925                      5.0        0.500               0.829                    0.294
agent::dr_reyna_holistic    0.0            0.716                      5.0        0.467               0.646                    0.077
   agent::theglowjournal    0.0            0.738                      3.0        0.700               0.046                    0.000
    agent::thriftymumof3    0.0

### The campaign, drawn

Node colour is the true label, node size is degree. What to look for: the agents forming a
visibly denser core than the organic periphery. That visual is the same evidence the GNN
is using, and it is what `NetworkGraph.jsx` renders in the Phase-3 dashboard.

In [19]:
viz.plot_network(
    campaign_nodes, campaign_edges, label_col="label",
    score_col=None, save_as=settings.paths.figures / "03_campaign_network.png",
)

if HAS_PYG:
    _scored = campaign_nodes.merge(
        _cf.loc[:, ["user_id", "score"]], on="user_id", how="left"
    )
    viz.plot_network(
        _scored, campaign_edges, label_col="label", score_col="score",
        save_as=settings.paths.figures / "03_campaign_network_scored.png",
    )

11:46:40 │ INFO    │ aegis.viz │ figure -> C:\Users\dabhi\Documents\Major-Project\Complete-project\reports\figures\03_campaign_network.png


11:46:40 │ INFO    │ aegis.viz │ figure -> C:\Users\dabhi\Documents\Major-Project\Complete-project\reports\figures\03_campaign_network_scored.png


## 9 · Persist for the fusion notebook

In [20]:
graph_scores = features.features.loc[:, ["user_id", "label", "split"]].copy()
graph_scores["rf_score"] = _rf.predict_proba(X_scaled)[:, 1]
if HAS_PYG:
    graph_scores["graph_score"] = gnn_probs
    graph_scores["graph_pred"] = (gnn_probs >= graph_threshold).astype(int)
else:
    graph_scores["graph_score"] = graph_scores["rf_score"]
    graph_scores["graph_pred"] = (graph_scores["rf_score"] >= 0.5).astype(int)

graph_scores = graph_scores.merge(
    features.features.loc[:, ["user_id", *features.feature_columns]], on="user_id", how="left"
)
iou.save_frame(graph_scores, settings.paths.processed / "graph_scores.parquet")

campaign_scores = campaign_features.features.loc[:, ["user_id", "label"]].copy()
campaign_scores["rf_score"] = _rf.predict_proba(_Xc)[:, 1]
if HAS_PYG:
    campaign_scores["graph_score"] = _cprobs
else:
    campaign_scores["graph_score"] = campaign_scores["rf_score"]
campaign_scores = campaign_scores.merge(
    campaign_features.features.loc[:, ["user_id", *features.feature_columns]],
    on="user_id", how="left",
)
iou.save_frame(campaign_scores, settings.paths.processed / "campaign_graph_scores.parquet")

if HAS_PYG:
    torch.save(
        {
            "state_dict": model.state_dict(),
            "in_channels": int(data.num_node_features),
            "hidden": int(GCFG.get("hidden_channels", 128)),
            "architecture": str(GCFG.get("architecture", "sage")),
            "num_layers": int(GCFG.get("num_layers", 2)),
            "dropout": float(GCFG.get("dropout", 0.4)),
            "feature_columns": list(features.feature_columns),
            "threshold": float(graph_threshold),
        },
        settings.paths.graph_model / "swarm_gnn.pt",
    )

import joblib

joblib.dump(scaler, settings.paths.graph_model / "feature_scaler.joblib")
joblib.dump(_rf, settings.paths.graph_model / "rf_features.joblib")

iou.save_json(
    {
        "dataset": GRAPH_NAME,
        "nodes": int(bundle.n_nodes),
        "edges": int(bundle.n_edges),
        "relations": sorted(edges["relation"].unique().tolist()),
        "label_assortativity": float(nx.attribute_assortativity_coefficient(G, "label")),
        "synchrony_window_seconds": WINDOW,
        "feature_columns": list(features.feature_columns),
        "separation": separation.to_dict("records"),
        "threshold": float(graph_threshold) if HAS_PYG else None,
        "tabular_ablation": {k: v.to_dict() for k, v in tabular.items()},
        "gnn_test": gnn_report.to_dict() if HAS_PYG else None,
        "ablation": {k: v.to_dict() for k, v in ablation.items()} if HAS_PYG else None,
        "campaign_transfer": {k: v.to_dict() for k, v in campaign_results.items()},
        "caveat": (
            "Cresci-2017 is ~98% label-assortative because its genuine and spambot "
            "populations were crawled separately. Treat the Cresci score as an upper "
            "bound and the synthetic-campaign transfer score as the realistic one."
        ),
    },
    settings.paths.graph_model / "graph_metrics.json",
)
print(f"artefacts -> {settings.paths.graph_model}")

11:46:40 │ INFO    │ aegis.io │ wrote graph_scores.parquet                   rows=2074     cols=22  (208.2 KB)


11:46:40 │ INFO    │ aegis.io │ wrote campaign_graph_scores.parquet          rows=48       cols=20  (16.3 KB)


artefacts -> C:\Users\dabhi\Documents\Major-Project\Complete-project\models\graph_model


## Summary

**Artefacts:** `models/graph_model/swarm_gnn.pt`, `feature_scaler.joblib`,
`rf_features.joblib`, `graph_metrics.json`; per-account scores in
`data/processed/graph_scores.parquet` and `campaign_graph_scores.parquet`;
figures in `reports/figures/03_*.png`.

**Read the results in this order:** the label assortativity in §3 (the context for
everything after) → §7's rewired-null ablation (did topology matter, or only features?) →
**§8's transfer to the 2026 campaign**, which is the number to quote.

**What this branch cannot do:** it needs a *population*. Given one account with no
interaction history it has nothing to work with, where the text branch still has an
opinion. The two branches fail in opposite directions, which is the entire argument for
fusing them rather than picking one.

→ **`04_hybrid_fusion_model.ipynb`**